# 01 — Select worst briefs

Pick the bottom-N papers from a judged [offline evaluation](../../docs/specs/paper-brief-evaluation-offline.md) run.

**Input:** `data/paper_brief_evaluation/{run_id}/03-evaluations.jsonl` and `02-briefs.jsonl`, plus sibling `corpus/` `.txt` files.

**Output:** `data/paper_brief_improvement/{run_id}/01-worst-briefs.jsonl` — bottom N papers with brief, evaluation, and corpus path.

See [paper-brief-improvement.md](../../docs/specs/paper-brief-improvement.md) step 1.

In [1]:
# Folder name under data/paper_brief_evaluation/ (e.g. "20260818T221210Z_gemma4-e4b").
# Leave empty to use the latest run that already has 03-evaluations.jsonl.
SOURCE_RUN_ID = ""

# How many lowest evaluation_score papers to select.
WORST_N = 5

In [2]:
from __future__ import annotations

import json
import re
from pathlib import Path

from IPython.display import Markdown, display


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
EVAL_PARENT = REPO_ROOT / "data" / "paper_brief_evaluation"
CORPUS_DIR = EVAL_PARENT / "corpus"
IMPROVEMENT_PARENT = REPO_ROOT / "data" / "paper_brief_improvement"

print(f"repo root: {REPO_ROOT}")
print(f"eval parent: {EVAL_PARENT}")
print(f"corpus dir: {CORPUS_DIR}")

repo root: /workspace
eval parent: /workspace/data/paper_brief_evaluation
corpus dir: /workspace/data/paper_brief_evaluation/corpus


In [3]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def doi_to_filename(doi: str) -> str:
    return doi.upper().replace("/", "_") + ".txt"


def resolve_source_run(run_id: str) -> Path:
    if run_id:
        d = EVAL_PARENT / run_id
        if not (d / "03-evaluations.jsonl").is_file():
            raise FileNotFoundError(f"No 03-evaluations.jsonl in {d}")
        return d
    candidates = sorted(
        (
            p.parent
            for p in EVAL_PARENT.glob("*/03-evaluations.jsonl")
            if _RUN_ID_PATTERN.match(p.parent.name)
        ),
        key=lambda d: d.name,
    )
    if not candidates:
        raise FileNotFoundError("No judged runs found under " + str(EVAL_PARENT))
    return candidates[-1]


source_run_dir = resolve_source_run(SOURCE_RUN_ID)
run_id = source_run_dir.name
print(f"source run: {run_id}")

source run: 20260818T221210Z_gemma4-e4b


In [4]:
assert isinstance(WORST_N, int) and WORST_N > 0, (
    f"WORST_N must be a positive integer, got {WORST_N!r}"
)

evals = load_jsonl(source_run_dir / "03-evaluations.jsonl")
briefs = load_jsonl(source_run_dir / "02-briefs.jsonl")

brief_by_doi: dict[str, dict] = {}
for row in briefs:
    if row.get("brief") is not None:
        brief_by_doi[row["doi"]] = row["brief"]

scored = [
    row for row in evals
    if isinstance(row.get("evaluation_score"), (int, float))
    and not isinstance(row.get("evaluation_score"), bool)
]
scored.sort(key=lambda r: (r["evaluation_score"], r["doi"]))

worst = scored[:WORST_N]
print(f"scored evaluations: {len(scored)}")
print(f"error evaluations (skipped): {len(evals) - len(scored)}")
print(f"selected bottom {len(worst)} papers")

scored evaluations: 96
error evaluations (skipped): 8
selected bottom 5 papers


In [5]:
output_rows: list[dict] = []
for row in worst:
    doi = row["doi"]
    corpus_file = CORPUS_DIR / doi_to_filename(doi)
    if not corpus_file.is_file():
        print(f"WARNING: corpus file missing for {doi}: {corpus_file}")
        continue
    brief = brief_by_doi.get(doi)
    if brief is None:
        print(f"WARNING: no brief found for {doi}")
        continue
    output_rows.append({
        "doi": doi,
        "evaluation_score": row["evaluation_score"],
        "evaluation": row["evaluation"],
        "brief": brief,
        "corpus_file": str(corpus_file.relative_to(REPO_ROOT)),
    })

print(f"output rows: {len(output_rows)}")

output rows: 5


In [6]:
out_dir = IMPROVEMENT_PARENT / run_id
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "01-worst-briefs.jsonl"

with out_path.open("w", encoding="utf-8") as fh:
    for row in output_rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"wrote {len(output_rows)} rows to {out_path.relative_to(REPO_ROOT)}")

wrote 5 rows to data/paper_brief_improvement/20260818T221210Z_gemma4-e4b/01-worst-briefs.jsonl


In [7]:
CRITERIA = ("faithfulness", "completeness", "conciseness", "topic_agnostic")

lines = ["| DOI | Score | " + " | ".join(CRITERIA) + " |"]
lines.append("| --- | ---: | " + " | ".join(["---:"] * len(CRITERIA)) + " |")
for row in output_rows:
    criterion_scores = []
    for c in CRITERIA:
        ev = row.get("evaluation", {})
        s = ev.get(c, {}).get("score", "–")
        criterion_scores.append(str(s))
    lines.append(
        f"| `{row['doi']}` | {row['evaluation_score']:.2f} | "
        + " | ".join(criterion_scores)
        + " |"
    )

display(Markdown("### Selected worst briefs\n\n" + "\n".join(lines)))

### Selected worst briefs

| DOI | Score | faithfulness | completeness | conciseness | topic_agnostic |
| --- | ---: | ---: | ---: | ---: | ---: |
| `10.1093/JME/TJAG095` | 4.00 | 2 | 5 | 4 | 5 |
| `10.3390/VACCINES14060499` | 4.00 | 2 | 5 | 4 | 5 |
| `10.64898/2026.05.10.722846` | 4.00 | 2 | 5 | 4 | 5 |
| `10.1016/J.APSB.2026.02.021` | 4.25 | 3 | 5 | 4 | 5 |
| `10.1038/S41467-026-73251-5` | 4.25 | 5 | 4 | 5 | 3 |